# 🏥 Análise de Intoxicações Exógenas do SINAN/SUS

**Aluna: Giedry Fernanda Trindade**

**Objetivo do notebook:** baixar a base de notificações de **Intoxicação Exógena (IEXO)** do SINAN
(Sistema de Informação de Agravos de Notificação) do SUS, tratá-la e exportar um arquivo
único e limpo para ser usado em **dashboards no Power BI**.

---

## 📋 O que este notebook faz

Cada passo é independente e bem identificado. Você pode rodar tudo do início ao fim,
ou voltar e re-executar um passo específico em uma células através do comando CTRL + ENTER.

| Passo | O que faz | Por que importa |
|-------|-----------|-----------------|
| 1 | Instala a biblioteca `pysus` | É o nosso conector oficial com o DataSUS |
| 2 | Importa bibliotecas e configura o ambiente | Prepara o terreno para as análises |
| 3 | Lista os anos disponíveis no SINAN-IEXO | Saber o intervalo de datas existentes antes de baixar |
| 4 | Baixa os dados de todos os anos (com retomada automática) | A base é grande. Se download cair no meio, não perde progresso |
| 5 | Inspeciona a estrutura dos dados baixados | Ver campos da base de dados |
| 6 | Converte campos de data para tipo `datetime` | Sem isso, não dá pra fazer análises temporais |
| 7 | Normaliza nomes de agentes tóxicos (regex) | "CLONAZEPAN" e "RIVOTRIL" viram **CLONAZEPAM** com expressões regulares |
| 8 | Calcula idade exata e faixa etária | Base usa código numérico esquisito para idade, tratamos isso |
| 9 | Traduz códigos numéricos para descrições | Ninguém entende `EVOLUCAO=3`. Mas "Óbito por intoxicação exógena", sim |
| 10 | Cria indicadores derivados (flags de óbito, suicídio, tempos) | Step para criar visualizações mais poderosas no Power BI |
| 12 | Cria tabelas de Estados e Municípios (API do IBGE) | Para ter nome + UF + região |
| 13 | Junta as tabelas geográficas com os dados do SINAN | Enriquece a base com geografia brasileira |
| 14 | Valida a qualidade da base tratada | Mostra completude, sanidade, distribuição |
| 15 | Exporta o CSV final pronto para Power BI | Arquivo único, codificação correta, ordem lógica |

---

## 📚 Fontes e referências

- **Dicionário de Dados oficial SINAN-IEXO v5**: <http://portalsinan.saude.gov.br/images/documentos/Agravos/iexog/DIC_DADOS_Intoxicacao_Exogena_v5.pdf>
- **Portal SINAN — Intoxicação Exógena**: <https://portalsinan.saude.gov.br/intoxicacao-exogena>
- **Biblioteca pysus**: <https://pysus.readthedocs.io/>
- **API de localidades do IBGE**: <https://servicodados.ibge.gov.br/api/docs/localidades>


## PASSO 1: Instalação da biblioteca `pysus`

A `pysus` é uma biblioteca que faz o download direto dos dados
do DataSUS (incluindo SINAN, SIM, SIH, SINASC, etc.) e já entrega como `DataFrame` do pandas.

> ⚠️ A célula abaixo só precisa rodar **uma vez por sessão** do Colab. Se você reiniciar
> o ambiente (`Ambiente → Reiniciar`), precisará rodar de novo.

In [ ]:
import importlib.metadata as md
import os
import subprocess
import sys

PACOTES = ["pysus==2.11.2"]

def _ambiente_ok() -> bool:
    """True se os pacotes estão instalados E o que roda em memória bate com o disco."""
    try:
        if md.version("pysus") != "2.11.2":
            return False
    except md.PackageNotFoundError:
        return False
    import numpy, pandas
    return (numpy.__version__ == md.version("numpy")
            and pandas.__version__ == md.version("pandas"))

if _ambiente_ok():
    print("✅ Ambiente consistente. Siga em frente.")
else:
    r = subprocess.run(
        ["uv", "pip", "install", "--python", sys.executable, "-q", *PACOTES],
        capture_output=True, text=True,
    )
    if r.returncode:
        print(r.stdout, r.stderr, sep="\n", file=sys.stderr)
        raise RuntimeError("Falha ao instalar dependências")
    print("Instalado. Reiniciando o kernel. Depois rode 'Ambiente → Executar tudo'.")
    os.kill(os.getpid(), 9)

## PASSO 2: Importação das bibliotecas e configurações iniciais

Aqui carregamos tudo de que vamos precisar:

- **pandas**: biblioteca para manipulação de tabelas (ou dataframes)
- **numpy**: cálculos numéricos vetorizados (mais rápido)
- **datetime**: trabalhar com datas
- **pysus**: baixar dados do SUS
- **nest_asyncio**: corrige um conflito entre o pysus (que usa `asyncio`) e o Google Colab
- **requests**: chamar a API do IBGE no Passo 12 para localizações geográficas

In [ ]:
# Biblioteca padrão
import gc
import os
import re
import time
from datetime import datetime

# Terceiros — científico/dados
import nest_asyncio
import numpy as np
import pandas as pd
import requests
import httpx

# Terceiros — DataSUS
from pysus import list_files, sinan

# Configurações globais
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
nest_asyncio.apply()


# ── Utilitários de memória ────────────────────────────────────────────────────
# A base tem ~2,5 M de linhas × 82 colunas. Em `object` dtype isso passa fácil
# dos 10 GB e estoura a RAM do Colab. Estes helpers mostram para onde a memória
# está indo e liberam o que já não é mais necessário.

def mem(rotulo: str = "") -> None:
    """Imprime a RAM atualmente em uso pelo processo."""
    with open("/proc/self/status") as f:
        kb = next(int(l.split()[1]) for l in f if l.startswith("VmRSS:"))
    print(f"🧠 RAM em uso: {kb / 1024**2:5.2f} GB   {rotulo}")


def ram_dfs(d: dict) -> float:
    """GB ocupados por um dicionário de DataFrames (conta o peso das strings)."""
    return sum(df.memory_usage(deep=True).sum() for df in d.values()) / 1024**3


def liberar(*nomes: str) -> None:
    """Descarta variáveis globais pesadas e força o coletor de lixo."""
    for n in nomes:
        globals().pop(n, None)
    gc.collect()


print("numpy :", np.__version__)
print("pandas:", pd.__version__)
print("✅ Bibliotecas carregadas com sucesso")
mem("(ambiente limpo)")

## PASSO 3: Quais anos estão disponíveis?

Antes de baixar, vamos perguntar ao DataSUS: "quais arquivos de Intoxicação Exógena
você tem?". A função `list_files` devolve a lista, e ordenamos por ano.

> 🔍 O parâmetro `group="IEXO"` é a **sigla oficial** da Intoxicação Exógena no SINAN.
> Outros agravos têm outras siglas (ex.: `DENG` = Dengue, `LEPT` = Leptospirose).

Obs. Foi necessário um pequeno FIX na chamada do list files

In [ ]:
if not hasattr(httpx.AsyncClient, "_pysus_original_init"):
    httpx.AsyncClient._pysus_original_init = httpx.AsyncClient.__init__

def _patched_init(self, *args, **kwargs):
    kwargs.setdefault(
        "timeout",
        httpx.Timeout(120.0, connect=30.0, read=120.0, write=30.0, pool=None),
    )
    httpx.AsyncClient._pysus_original_init(self, *args, **kwargs)

httpx.AsyncClient.__init__ = _patched_init

def list_files_retry(*args, max_tentativas=5, **kwargs):
    for n in range(1, max_tentativas + 1):
        try:
            return list_files(*args, **kwargs)
        except (httpx.ReadTimeout, httpx.ConnectTimeout,
                httpx.RemoteProtocolError, httpx.NetworkError) as e:
            if n == max_tentativas:
                raise
            espera = 2 ** n
            print(f"⚠️  tentativa {n}/{max_tentativas} falhou ({type(e).__name__}); "
                  f"aguardando {espera}s e tentando novamente…")
            time.sleep(espera)

arquivos = list_files_retry("SINAN", group="IEXO")
arquivos.sort_values(by="year")

## PASSO 4: Download em lote com retomada automática

Vamos baixar **todos os anos** desde 2006 até hoje. A base é grande, e o download pode
falhar em algum ano (servidor instável, timeout, etc.). Por isso o código:

1. **Pula anos já baixados**: se a célula já rodou e você está re-executando, ele não
   refaz o download (poupa banda e tempo).
2. **Guarda as falhas**: se algum ano falhou, ele aparece no final. Basta **rodar a
   célula de novo** que ele tenta só os que faltaram.
3. **Mostra progresso**: você vê em tempo real quantos registros vieram por ano.

> ⏱️ Estimativa: 5–20 min na primeira execução (depende da velocidade do DataSUS).
> Se travar, espere um pouco antes de cancelar! O servidor às vezes é lento, mas responde.

> 🧠 **Por que um dicionário `dfs`?** Cada ano vem como um DataFrame separado e podem
> ter colunas levemente diferentes entre versões antigas e novas. Mantendo separados,
> conseguimos tratar cada base anual sem quebrar as outros. **Só concatenamos no final.**

In [ ]:
# Cria MountPoint

from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Pasta onde cada ano vai ficar salvo em parquet
PASTA_DRIVE = Path("/content/drive/MyDrive/SINAN_IEXO")
PASTA_DRIVE.mkdir(parents=True, exist_ok=True)

print(f"✅ Drive montado. Cache em: {PASTA_DRIVE}")

In [ ]:
# PASSO 4 — Download em lote com cache no Drive

ANO_INICIO  = 2006
ANO_FIM     = 2025            # 2026 ainda é parcial (arquivo de 31/07); suba quando fechar
ANOS_FORCAR = {ANO_FIM}       # rebaixa mesmo se já houver parquet no Drive (retificações)

if "dfs" not in dir():
    dfs = {}

# 1) Aproveita o que já está no Drive
for arq in sorted(PASTA_DRIVE.glob("sinan_iexo_*.parquet")):
    ano = int(arq.stem.split("_")[-1])
    if ANO_INICIO <= ano <= ANO_FIM and ano not in ANOS_FORCAR and ano not in dfs:
        dfs[ano] = pd.read_parquet(arq)
        print(f"💾 {ano} — Drive ({len(dfs[ano]):,} registros)")

# 2) Baixa o que falta
falhas = {}
for ano in range(ANO_INICIO, ANO_FIM + 1):
    if ano in dfs:
        continue
    try:
        df_ano = sinan(disease="IEXO", year=ano, as_dataframe=True, show_progress=False)
    except Exception as e:
        falhas[ano] = f"{type(e).__name__}: {e}"
        continue

    if not isinstance(df_ano, pd.DataFrame):
        falhas[ano] = f"retorno {type(df_ano).__name__}, não DataFrame"
        continue
    if df_ano.empty:
        falhas[ano] = "0 registros"
        continue

    dfs[ano] = df_ano
    df_ano.to_parquet(PASTA_DRIVE / f"sinan_iexo_{ano}.parquet", index=False)
    print(f"⬇️  {ano} — DataSUS ({len(df_ano):,} registros)")

# 3) Relatório e trava
for ano, motivo in sorted(falhas.items()):
    print(f"❌ {ano}: {motivo}")

faltando = sorted(set(range(ANO_INICIO, ANO_FIM + 1)) - set(dfs))
print(f"\n📦 {len(dfs)} anos | {sum(map(len, dfs.values())):,} registros")
if faltando:
    raise RuntimeError(f"Anos ausentes: {faltando}. Rode a célula de novo para tentar só eles.")

print(f"   Peso de `dfs` em memória: {ram_dfs(dfs):.2f} GB")
mem("(após o download)")

## PASSO 5 — Mapeamento estrutural da base

Antes de tratar, precisamos entender se a base é **homogênea ao longo dos anos**
,ou seja, se as colunas, tipos e volumes são consistentes de 2006 até hoje.

A análise abaixo detecta automaticamente três tipos de anomalia:

| Tipo | O que significa | Impacto no tratamento |
|---|---|---|
| **Ano atípico** | Volume ou número de colunas muito fora da mediana | Pode distorcer séries históricas |
| **Coluna instável** | Presente em alguns anos, ausente em outros | Campos derivados dela serão `NaN` nos anos que faltam |
| **Dtype inconsistente** | Mesmo campo, tipo diferente entre anos | Concatenar diretamente pode causar erro silencioso |

O output é **enxuto por design**: só aparece o que está errado. Se tudo estiver ok,
cada seção imprime uma única linha de confirmação.

In [ ]:
# PASSO 5 — Mapeamento estrutural da base

anos = sorted(dfs.keys())
cols_por_ano  = {a: set(dfs[a].columns)       for a in anos}
tipos_por_ano = {a: dfs[a].dtypes.to_dict()   for a in anos}
todas_cols    = sorted(set().union(*cols_por_ano.values()))

n_registros = {a: len(dfs[a])          for a in anos}
n_colunas   = {a: len(dfs[a].columns)  for a in anos}

sep = "=" * 60

# ── 1. VOLUME POR ANO ────────────────────────────────────────
print(sep)
print("1. VOLUME POR ANO")
print(sep)

mediana_reg = np.median(list(n_registros.values()))
mediana_col = np.median(list(n_colunas.values()))
LIMIAR      = 0.5   # ano com < 50 % da mediana é sinalizado como atípico

print(f"{'ANO':>6}  {'REGISTROS':>12}  {'COLUNAS':>8}  {'OBS':}")
print("-" * 50)
for a in anos:
    flag = ""
    if n_registros[a] < mediana_reg * LIMIAR:
        flag += "  ⚠️ volume baixo"
    if n_colunas[a] != mediana_col:
        flag += f"  ⚠️ {int(n_colunas[a] - mediana_col):+d} col vs mediana"
    print(f"{a:>6}  {n_registros[a]:>12,}  {n_colunas[a]:>8}{flag}")

print("-" * 50)
print(f"{'TOTAL':>6}  {sum(n_registros.values()):>12,}")
print(f"\n  Mediana de registros: {mediana_reg:,.0f}  |  Mediana de colunas: {mediana_col:.0f}")


# ── 2. COLUNAS INSTÁVEIS ─────────────────────────────────────
print(f"\n{sep}")
print("2. COLUNAS INSTÁVEIS (ausentes em pelo menos 1 ano)")
print(sep)

instáveis = [c for c in todas_cols
             if not all(c in cols_por_ano[a] for a in anos)]

if not instáveis:
    print("  ✅ Todas as colunas estão presentes em 100 % dos anos.")
else:
    # Agrupa por padrão de presença para não listar linha por linha
    padroes: dict[tuple, list] = {}
    for c in instáveis:
        padrao = tuple(c in cols_por_ano[a] for a in anos)
        padroes.setdefault(padrao, []).append(c)

    for padrao, cols in padroes.items():
        ausentes  = [a for a, p in zip(anos, padrao) if not p]
        presentes = [a for a, p in zip(anos, padrao) if p]
        primeira_aparicao = presentes[0] if presentes else "—"

        if len(ausentes) == 1:
            descricao = f"ausente só em {ausentes[0]}"
        elif ausentes == list(range(ausentes[0], ausentes[-1] + 1)):
            descricao = f"ausente de {ausentes[0]} a {ausentes[-1]}"
        else:
            descricao = f"ausente em {ausentes}"

        prefixo = "⭐ estreia em" if ausentes[0] < primeira_aparicao else "⚠️ "
        print(f"\n  {prefixo} {primeira_aparicao} — {descricao}:")
        for c in cols:
            print(f"     {'·' if ausentes else '✓'} {c}")


# ── 3. VARIAÇÃO DE DTYPE ──────────────────────────────────────
print(f"\n{sep}")
print("3. VARIAÇÃO DE DTYPE (só colunas onde o tipo mudou)")
print(sep)

try:
    from itertools import pairwise
except ImportError:
    def pairwise(it):
        a = None
        for b in it:
            if a is not None:
                yield a, b
            a = b

mudancas_globais = {}   # col → {(tipo_ant, tipo_atu): [pares de ano]}
for a_ant, a_atu in pairwise(anos):
    comuns = cols_por_ano[a_ant] & cols_por_ano[a_atu]
    for col in comuns:
        t_ant = str(tipos_por_ano[a_ant][col])
        t_atu = str(tipos_por_ano[a_atu][col])
        if t_ant != t_atu:
            entrada = mudancas_globais.setdefault(col, {})
            entrada.setdefault((t_ant, t_atu), []).append((a_ant, a_atu))

if not mudancas_globais:
    print("  ✅ Nenhum dtype mudou entre anos consecutivos.")
else:
    for col, transicoes in sorted(mudancas_globais.items()):
        print(f"\n  🔄 {col}")
        for (t_ant, t_atu), pares in transicoes.items():
            anos_str = ", ".join(f"{a}→{b}" for a, b in pares)
            print(f"     {t_ant} → {t_atu}  (em: {anos_str})")


# ── 4. SUMÁRIO EXECUTIVO ──────────────────────────────────────
print(f"\n{sep}")
print("4. SUMÁRIO EXECUTIVO")
print(sep)

anos_atipicos = [a for a in anos if n_registros[a] < mediana_reg * LIMIAR
                 or n_colunas[a] != mediana_col]
print(f"  Anos analisados:       {len(anos)}")
print(f"  Anos atípicos:         {anos_atipicos or '—'}")
print(f"  Colunas instáveis:     {len(instáveis) or '—'}")
print(f"  Colunas c/ dtype var.: {len(mudancas_globais) or '—'}")

if anos_atipicos:
    print(f"\n  💡 Sugestão: considere excluir {anos_atipicos} das análises")
    print(f"     de série histórica ou tratá-los separadamente.")

## PASSO 5b — Compactação de dtypes

*(passo adicionado na otimização de memória; não altera nenhum valor)*

O parquet do DataSUS chega quase todo como `object`: cada célula é um objeto
string solto em Python, com dezenas de bytes de overhead **por célula**. São
2,5 M de linhas × 82 colunas, o que sozinho já consome vários GB antes de
qualquer tratamento.

Quase todas essas colunas são códigos de baixa cardinalidade (algumas dezenas
de valores distintos repetidos milhões de vezes). O tipo `category` guarda cada
valor uma única vez e usa inteiros pequenos como ponteiro. Os valores continuam
exatamente os mesmos; muda só a representação em memória.

Este passo roda **depois** do PASSO 5 de propósito: o mapeamento estrutural
precisa enxergar os dtypes originais do arquivo para o relatório fazer sentido.

Ficam de fora as colunas de data (o PASSO 6 precisa delas como texto para o
`pd.to_datetime`) e `NU_IDADE_N` (o PASSO 8 faz `pd.to_numeric` nela).

In [ ]:
# PASSO 5b — Compactação de dtypes (só muda o tipo, nunca o valor)

NAO_COMPACTAR = {
    "DT_NOTIFIC", "DT_SIN_PRI", "DT_NASC", "DT_INVEST",
    "DT_OBITO", "DT_ENCERRA", "DT_DIGITA", "NU_IDADE_N",
}
LIMIAR_CARDINALIDADE = 0.5   # vira category se os distintos forem < 50% das linhas


def compactar(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        if col in NAO_COMPACTAR:
            continue
        s = df[col]
        # Só texto vira category. Downcast numérico (float64 → float32, por ex.)
        # ficou de fora de propósito: mudaria como o número é escrito no CSV
        # final, e o combinado é não alterar o produto do notebook.
        if s.dtype == "object" or pd.api.types.is_string_dtype(s):
            if len(s) and s.nunique(dropna=False) / len(s) < LIMIAR_CARDINALIDADE:
                df[col] = s.astype("category")
    return df


antes = ram_dfs(dfs)
for ano in sorted(dfs):
    dfs[ano] = compactar(dfs[ano])
gc.collect()
depois = ram_dfs(dfs)

print(f"🗜️  `dfs`: {antes:.2f} GB → {depois:.2f} GB  "
      f"({(1 - depois / antes) * 100:.0f}% menor)")
mem("(após compactar)")

## PASSO 6 — Conversão de datas e descontinuação de `DT_NASC`

O DataSUS entrega todas as datas como **texto** (`"20230515"` ou `"2023-05-15"`).
Enquanto permanecerem como `object`, nenhum cálculo de tempo é possível.

A conversão usa `errors="coerce"`: qualquer valor que não seja uma data válida
(campos vazios, `"99999999"`, `"00000000"`, lixo de digitação) vira `NaT` em
silêncio. O diagnóstico abaixo torna esse silêncio **visível**: mostra quantos
valores foram convertidos com sucesso, quantos viraram `NaT` e **quais eram os
valores originais problemáticos** — útil para decidir se vale tentar recuperá-los.

`DT_NASC` é **descontinuada a partir de 2007** (presente apenas em 2006, com 158
registros). Como a base de 2006 já é atípica em volume e estrutura, manter a
coluna só acumularia `NaT` em todos os demais anos. Ela é removida após a
conversão e a idade passa a ser calculada exclusivamente via `NU_IDADE_N`.

In [ ]:
# PASSO 6 — Conversão de datas e descontinuação de DT_NASC

COLUNAS_DATA  = ["DT_NOTIFIC", "DT_SIN_PRI", "DT_NASC", "DT_INVEST",
                 "DT_OBITO", "DT_ENCERRA", "DT_DIGITA"]
COL_DESCONT   = "DT_NASC"     # presente só em 2006; descontinuada a partir de 2007
TOP_INVALIDOS = 5             # quantos valores problemáticos mostrar por coluna

def converter_datas(df: pd.DataFrame) -> pd.DataFrame:
    """Converte colunas de data para datetime; erros viram NaT.

    OTIMIZAÇÃO: removido o `df = df.copy()` do início. Ele duplicava as 82
    colunas do ano só para reescrever 7 delas. Os valores originais das colunas
    de data já ficam preservados em `dfs_brutos` logo abaixo, então a cópia
    integral era desperdício puro.
    """
    for col in COLUNAS_DATA:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

# ── Aplica a conversão ────────────────────────────────────────────────────────
# OTIMIZAÇÃO (a maior do notebook): antes era `dfs_brutos = dfs.copy()`, que
# mantinha a base bruta INTEIRA (82 colunas × 20 anos) viva na RAM até o final,
# em paralelo com a base tratada. O diagnóstico abaixo só lê as colunas de data,
# então guardamos apenas elas, e descartamos tudo no fim da célula.
dfs_brutos = {
    ano: df[[c for c in COLUNAS_DATA if c in df.columns]].copy()
    for ano, df in dfs.items()
}

# OTIMIZAÇÃO: laço com reatribuição no lugar do dict comprehension. O
# comprehension montava um dicionário novo enquanto o antigo seguia inteiro na
# memória, dobrando o pico. Aqui cada ano antigo é liberado assim que o novo
# toma o lugar dele.
for ano in sorted(dfs):
    dfs[ano] = converter_datas(dfs[ano])
gc.collect()

# ── Diagnóstico por coluna ────────────────────────────────────────────────────
sep = "=" * 60
print(sep)
print("DIAGNÓSTICO DE CONVERSÃO DE DATAS")
print(sep)

for col in COLUNAS_DATA:
    anos_com_col = [a for a in sorted(dfs.keys()) if col in dfs[a].columns]
    if not anos_com_col:
        continue

    # Agrega estatísticas de todos os anos
    total      = sum(len(dfs[a])                        for a in anos_com_col)
    convertidos = sum(dfs[a][col].notna().sum()          for a in anos_com_col)
    nats        = sum(dfs[a][col].isna().sum()           for a in anos_com_col)

    pct_ok  = convertidos / total * 100 if total else 0
    pct_nat = nats        / total * 100 if total else 0

    print(f"\n  {col}")
    print(f"    Anos com a coluna : {anos_com_col}")
    print(f"    Total de células  : {total:>10,}")
    print(f"    ✓ Convertidos     : {convertidos:>10,}  ({pct_ok:.1f}%)")
    print(f"    ✗ Virou NaT       : {nats:>10,}  ({pct_nat:.1f}%)")

    # Mostra os valores brutos mais frequentes que viraram NaT
    if nats > 0:
        invalidos = (
            pd.concat(
                [
                    dfs_brutos[a][col][dfs[a][col].isna()]
                    for a in anos_com_col
                    if col in dfs_brutos[a].columns
                ]
            )
            .astype(str)
            .str.strip()
            .replace("", "<vazio>")
            .replace("nan", "<nulo>")
            .value_counts()
        )
        print(f"    Top valores inválidos (virou NaT):")
        for val, cnt in invalidos.head(TOP_INVALIDOS).items():
            print(f"      '{val}' → {cnt:,}×")
        if len(invalidos) > TOP_INVALIDOS:
            print(f"      … e mais {len(invalidos) - TOP_INVALIDOS} valor(es) distintos")
        del invalidos

# ── Remove DT_NASC (descontinuada) ───────────────────────────────────────────
print(f"\n{sep}")
print(f"REMOÇÃO DE {COL_DESCONT}")
print(sep)

anos_removidos = []
for ano in sorted(dfs.keys()):
    if COL_DESCONT in dfs[ano].columns:
        dfs[ano] = dfs[ano].drop(columns=[COL_DESCONT])
        anos_removidos.append(ano)

if anos_removidos:
    print(f"  ✖ {COL_DESCONT} removida de: {anos_removidos}")
    print(f"  ℹ️  Coluna presente só em {anos_removidos} (descontinuada no SINAN a partir de 2007).")
    print(f"     Idade passa a ser calculada exclusivamente via NU_IDADE_N.")
else:
    print(f"  ℹ️  {COL_DESCONT} não encontrada em nenhum ano — nada a remover.")

# OTIMIZAÇÃO: o diagnóstico acabou, então a base bruta vai embora agora.
liberar("dfs_brutos")

print(f"\n✅ Conversão e limpeza de datas concluídas.")
mem("(após o PASSO 6)")

## PASSO 7: Normalização de nomes de agentes tóxicos

O campo `AGENTE_1` (princípio ativo / nome comercial do agente da intoxicação) é de
**texto livre**. Isso significa que o mesmo agente aparece com várias grafias:

- `CLONAZEPAM`, `CLONAZEPAN`, `CLOMAZEPAM`, `CLONAZEPAM 2MG`...
- `RIVOTRIL`, `RIVOTRYL`, `RIVOLTRIL`, `RIV0TRIL` (zero no lugar de O!)

Se a gente não normalizar, no Power BI vão aparecer dezenas de "agentes diferentes"
quando, na verdade, é tudo a mesma coisa.

A função abaixo usa **expressões regulares (regex)** para capturar todas essas variantes
e unificar em um único nome canônico — neste caso, `CLONAZEPAM`.

> 💡 **Como expandir:** quer normalizar mais agentes (paracetamol, dipirona, etc.)?
> Adicione uma nova entrada no dicionário `NORMALIZACAO_AGENTES`. O padrão pode ser
> simples (`r".*PARACETAMOL.*"`) ou complexo como o exemplo abaixo.

In [ ]:
# Dicionário exaustivo de normalização de agentes tóxicos
# Inclui medicamentos, drogas de abuso, praguicidas e produtos químicos comuns no SINAN

NORMALIZACAO_AGENTES = {
    # --- BENZODIAZEPÍNICOS E ANSIOLÍTICOS ---
    "CLONAZEPAM": r".*CLO[NM][AE][ZS][EA]P[AO][MN].*|.*R[IY]L?V[O0](V[O0])?L?TR[IY]L.*",
    "DIAZEPAM": r".*DIA[ZS]EP[AO]M.*|.*VALIUM.*",
    "ALPRAZOLAM": r".*ALPRA[ZS]OL[AO]M.*|.*FRONTAL.*",
    "BROMAZEPAM": r".*BROMA[ZS]EP[AO]M.*|.*LEXOTAN.*",
    "LORAZEPAM": r".*LORA[ZS]EP[AO]M.*|.*LORAX.*",
    "MIDAZOLAM": r".*MIDA[ZS]OL[AO]M.*|.*DORMONID.*",

    # --- ANTIDEPRESSIVOS E ANTIPSICÓTICOS ---
    "SERTRALINA": r".*S[EI]R?TRALIN[AE].*|.*ZOLOFT.*",
    "AMITRIPTILINA": r".*AM[EIY]TRIPTILIN[AE].*",
    "FLUOXETINA": r".*FLU[OX]{1,2}ETIN[AE].*|.*PROZAC.*",
    "CITALOPRAM": r".*CITALOPRA[MN].*|.*ESCITALOPRA[MN].*",
    "HALOPERIDOL": r".*HALO.*PERIDOL.*|.*HALDOL.*",
    "QUETIAPINA": r".*QUETIAPIN[AE].*",
    "RISPERIDONA": r".*RISPERIDON[AE].*",

    # --- ANALGÉSICOS E ANTI-INFLAMATÓRIOS ---
    "PARACETAMOL": r".*PARA[CS]ETAM[OU]L.*|.*TYLENOL.*",
    "DIPIRONA": r".*DIPIRON[AE].*|.*METAMIZOL.*|.*NOVALGINA.*",
    "IBUPROFENO": r".*IBUPROFEN[OU].*|.*ADVIL.*",
    "DICLOFENACO": r".*DICLOFENAC[OU].*|.*VOLTAREN.*|.*CATAFLAM.*",
    "NIMESULIDA": r".*NIMESULID[AE].*",

    # --- DROGAS DE ABUSO ---
    "COCAINA": r".*COCAIN[AE].*",
    "CRACK": r".*CRACK.*",
    "MACONHA": r".*MACONHA.*|.*CANNABIS.*|.*MARI[JH]UANA.*",
    "ECSTASY": r".*ECSTASY.*|.*MDMA.*",
    "LSD": r".*LSD.*|.*ACIDO.*LISERGICO.*",
    "ANFETAMINA": r".*ANFETAMIN[AE].*|.*REBITE.*",
    "ALCOOL": r".*ALC[OU]OL.*|.*ETANOL.*|.*CACHAC[AO].*|.*PINGA.*|.*VINHO.*|.*CERVEJA.*|.*VODKA.*",

    # --- PRAGUICIDAS E RATICIDAS ---
    "CHUMBINHO": r".*CHUMBINHO.*|.*CHUBINHO.*|.*ALDICARB.*",
    "RATICIDA": r".*RATICID[AO].*|.*VENENO.*RAT[OU].*",
    "ROUNDUP": r".*ROUNDUP.*|.*RANDAP.*|.*ROUNDAP.*|.*GLIFOSATO.*",
    "PARAQUAT": r".*PARAQUAT.*|.*GRAMOCIL.*",

    # --- DOMISSANITÁRIOS E QUÍMICOS ---
    "SODA CAUSTICA": r".*SODA.*CAUSTIC[AO].*|.*HIDROXIDO.*SODIO.*",
    "AGUA SANITARIA": r".*AGUA.*SANITAR[IA].*|.*QBOA.*|.*QUIBOA.*|.*KIBOA.*|.*HIPOCLORITO.*",
    "DETERGENTE": r".*DETERGENTE.*|.*SABAO.*",
    "QUEROSENE": r".*QUEROSENE.*|.*QUEROSINE.*",
    "GASOLINA": r".*GASOLINA.*",
    "NAFTALINA": r".*NAFTALIN[AE].*",

    # --- OUTROS COMUNS ---
    "MONOXIDO DE CARBONO": r".*MONO[TX]IDO.*CARBONO.*|.*FUMACA.*",
    "ESCORPIAO": r".*ESCORPI[AO].*", # Embora seja animal peçonhento, às vezes aparece aqui
    "ARANHA": r".*ARANHA.*",
    "SERPENTE": r".*SERPENTE.*|.*COBRA.*"
}

print(f"✅ Dicionário exaustivo carregado com {len(NORMALIZACAO_AGENTES)} categorias.")

In [ ]:
def otimizar_agentes_vapt_vupt(dfs_dict):
    # 1. Compilar um regex único para todos os agentes (Muito mais rápido)
    patterns = []
    for nome, padrao in NORMALIZACAO_AGENTES.items():
        clean_pattern = padrao.replace('(?i)', '')
        patterns.append(f"(?P<{nome.replace(' ', '_')}>{clean_pattern})")

    combined_regex = re.compile("|".join(patterns), flags=re.IGNORECASE)

    def get_canonical_name(text):
        if pd.isna(text) or text == "": return text
        match = combined_regex.search(text)
        if match:
            return match.lastgroup.replace('_', ' ')
        return text

    print("⏳ Iniciando normalização otimizada...")
    for ano in sorted(dfs_dict.keys()):
        df = dfs_dict[ano]
        cols_agente = [c for c in ["AGENTE_1", "AGENTE_2", "AGENTE_3"] if c in df.columns]
        for col in cols_agente:
            serie_original = df[col].astype("string").str.upper().str.strip().fillna("")
            valores_unicos = serie_original.unique()
            mapa_substituicao = {val: get_canonical_name(val) for val in valores_unicos if val != ""}
            df[col] = serie_original.map(mapa_substituicao).fillna(serie_original)
            df[col] = df[col].astype("category")
            # OTIMIZAÇÃO: as séries intermediárias (uma cópia em `string` dtype da
            # coluna inteira, mais o resultado do map) são grandes. Descartar já
            # aqui evita que fiquem vivas enquanto o próximo ano é processado.
            del serie_original, valores_unicos, mapa_substituicao
        gc.collect()
        print(f"  ✅ {ano} concluído")
    return dfs_dict

dfs = otimizar_agentes_vapt_vupt(dfs)
print("\n🚀 Normalização concluída!")
print(f"   Peso de `dfs`: {ram_dfs(dfs):.2f} GB")
mem("(após o PASSO 7)")

## PASSO 8: Idade e Faixa Etária

A base **não possui data de nascimento** (`DT_NASC` foi descontinuada no SINAN-IEXO
a partir de 2007 e chegou 100% vazia no único ano em que existia). Toda a idade é
calculada exclusivamente a partir de `NU_IDADE_N`.

### Como o SINAN codifica a idade

| Primeiro dígito | Unidade | Exemplo | Leitura |
|:-:|---|---|---|
| `1` | Horas | `1018` | 18 horas |
| `2` | Dias  | `2012` | 12 dias  |
| `3` | Meses | `3008` | 8 meses  |
| `4` | Anos  | `4035` | 35 anos  |

### O que produzimos

| Coluna | Tipo | Detalhe |
|---|---|---|
| `IDADE_ANOS` | `Int64` | Inteiro; para unidades < ano, valor aproximado |
| `IDADE_DESCRICAO` | `object` | Texto legível, ex.: `"8 MESES"` ou `"35 ANOS"` |
| `FAIXA_ETARIA` | `object` | Categórica — veja faixas abaixo |

`IDADE_MESES` e `IDADE_DIAS` **não são criadas**: sem `DT_NASC`, os registros em
anos só chegam em anos inteiros e os registros em meses/dias representam menos de
**[% calculada em tempo real abaixo]** da base — e são quase exclusivamente
crianças pequenas, como o diagnóstico confirma.

### Faixas etárias

Seguem o padrão do Ministério da Saúde adaptado para intoxicação exógena:

| Faixa | Critério |
|---|---|
| `NEONATO` | unidade = horas **ou** < 28 dias |
| `LACTENTE` | 28 dias a < 2 anos |
| `CRIANÇA` | 2 a 11 anos |
| `ADOLESCENTE` | 12 a 17 anos |
| `ADULTO JOVEM` | 18 a 39 anos |
| `ADULTO` | 40 a 59 anos |
| `IDOSO` | 60 anos ou mais |
| `NÃO INFORMADO` | `NU_IDADE_N` ausente ou inválido |

In [ ]:
# PASSO 8a — Diagnóstico de NU_IDADE_N (rode antes de commitar o tratamento)

# OTIMIZAÇÃO: antes esta célula fazia `df_diag = pd.concat(dfs.values())`, ou
# seja, materializava a base inteira (82 colunas × 2,5 M linhas) na RAM só para
# ler UMA coluna — e `df_diag` continuava vivo pelo resto do notebook. Agora
# concatenamos apenas NU_IDADE_N. O índice resultante é o mesmo RangeIndex
# que o concat produzia, então todos os cálculos abaixo seguem idênticos.
raw_all = pd.concat(
    [pd.to_numeric(dfs[a]["NU_IDADE_N"], errors="coerce") for a in sorted(dfs)],
    ignore_index=True,
)
total   = len(raw_all)
und_all = (raw_all // 1000).astype("float64")
val_all = (raw_all  % 1000).astype("float64")

sep = "=" * 55
print(sep)
print("DIAGNÓSTICO — NU_IDADE_N")
print(sep)

# Distribuição por unidade
for cod, label in {1: "Horas", 2: "Dias", 3: "Meses", 4: "Anos"}.items():
    n = (und_all == cod).sum()
    print(f"  Unidade {cod} ({label:<6}): {n:>10,}  ({n/total*100:.2f}%)")

n_inv = und_all.isna().sum() + (~und_all.isin([1, 2, 3, 4])).sum()
print(f"  Inválido / nulo  : {n_inv:>10,}  ({n_inv/total*100:.2f}%)")

# Sub-anuais
mask_sub = und_all.isin([1, 2, 3])
n_sub    = mask_sub.sum()
print(f"\n  Sub-anuais (h+d+m): {n_sub:>9,}  ({n_sub/total*100:.2f}% da base)")

if n_sub > 0:
    # Faixa etária aproximada dos sub-anuais para confirmar que são crianças
    dias_sub = pd.Series(np.nan, index=raw_all.index)
    dias_sub = dias_sub.where(und_all != 1, val_all / 24)
    dias_sub = dias_sub.where(und_all != 2, val_all)
    dias_sub = dias_sub.where(und_all != 3, val_all * 30.44)

    faixa_sub = pd.cut(
        dias_sub[mask_sub],
        bins  = [-1, 27, 365.25 * 2, np.inf],
        labels= ["NEONATO (<28d)", "LACTENTE (28d–2a)", "≥2 anos (dado suspeito)"],
    )
    print(f"\n  Composição dos sub-anuais:")
    for faixa, cnt in faixa_sub.value_counts().items():
        print(f"    {faixa:<30}: {cnt:>8,}  ({cnt/n_sub*100:.1f}%)")

    # Mostra os valores brutos mais frequentes de sub-anuais ≥ 2 anos (suspeitos)
    suspeitos = dias_sub[mask_sub & (dias_sub >= 365.25 * 2)]
    if len(suspeitos) > 0:
        print(f"\n  ⚠️  {len(suspeitos):,} sub-anuais com idade ≥ 2 anos — possível erro de digitação.")
        raw_sus = raw_all[mask_sub & (dias_sub >= 365.25 * 2)]
        print(f"  Top valores brutos:")
        for v, c in raw_sus.value_counts().head(5).items():
            print(f"    NU_IDADE_N={v}  → {c:,}×")

# OTIMIZAÇÃO: o diagnóstico é descartável. Sem isto, sete séries de 2,5 M de
# linhas ficariam no escopo global até o fim do notebook.
liberar("raw_all", "und_all", "val_all", "mask_sub", "dias_sub",
        "faixa_sub", "suspeitos", "raw_sus")
mem("(após o PASSO 8a)")

In [ ]:
# PASSO 8b — Enriquecimento de idade (aplica no dfs)

def enriquecer_idade(df: pd.DataFrame) -> pd.DataFrame:
    if "NU_IDADE_N" not in df.columns:
        return df

    # Trabalha com arrays numpy — sem cópias do df inteiro
    raw     = pd.to_numeric(df["NU_IDADE_N"], errors="coerce").to_numpy(dtype="float64")
    unidade = raw // 1000
    valor   = raw  % 1000

    # ── Idade em anos ─────────────────────────────────────────────────────
    anos = np.where(unidade == 4, valor,
           np.where(unidade == 3, valor / 12,
           np.where(unidade == 2, valor / 365.25,
           np.where(unidade == 1, valor / 8766,
           np.nan))))

    # OTIMIZAÇÃO: a versão anterior fazia
    #     np.where(np.isnan(anos), pd.NA, np.round(anos).astype("int64"))
    # que (a) castava NaN para int64, disparando o RuntimeWarning "invalid value
    # encountered in cast" a cada ano, e (b) construía um array `object` de 2,5 M
    # de elementos só para depois virar Int64. Aqui montamos o IntegerArray
    # direto a partir dos valores e da máscara. O resultado é idêntico.
    nulo      = np.isnan(anos)
    idade_int = np.zeros(len(df), dtype="int64")
    idade_int[~nulo] = np.round(anos[~nulo]).astype("int64")
    df["IDADE_ANOS"] = pd.arrays.IntegerArray(idade_int, nulo.copy())

    # ── Descrição legível ─────────────────────────────────────────────────
    _sufixo = {
        4: ("ANO",  "ANOS"),
        3: ("MÊS",  "MESES"),
        2: ("DIA",  "DIAS"),
        1: ("HORA", "HORAS"),
    }
    v_int = np.where(np.isnan(valor), 0, valor).astype("int64")
    u_int = np.where(np.isnan(unidade), 0, unidade).astype("int64")

    desc = np.empty(len(df), dtype=object)
    desc[:] = None
    for cod, (sing, plur) in _sufixo.items():
        mask = (u_int == cod) & ~np.isnan(raw)
        if not mask.any():
            continue
        vals = v_int[mask]
        # OTIMIZAÇÃO: antes era uma f-string por linha, criando ~2,4 M de objetos
        # str distintos. Como só existem algumas centenas de valores possíveis,
        # montamos a tabela uma vez e reaproveitamos as mesmas strings.
        tabela = {v: f"{v} {sing if v == 1 else plur}" for v in np.unique(vals)}
        desc[mask] = [tabela[v] for v in vals]

    # OTIMIZAÇÃO: category. São ~500 rótulos distintos em 2,5 M de linhas.
    df["IDADE_DESCRICAO"] = pd.Categorical(desc)

    # ── Faixa etária ──────────────────────────────────────────────────────
    dias = np.where(unidade == 1, valor / 24,
           np.where(unidade == 2, valor,
           np.where(unidade == 3, valor * 30.44,
           np.where(unidade == 4, valor * 365.25,
           np.nan))))

    faixa = np.full(len(df), "NÃO INFORMADO", dtype=object)
    # Aplica do mais específico para o mais geral (última condição verdadeira vence)
    faixa[anos  >= 60]                         = "IDOSO"
    faixa[anos  <  60]                         = "ADULTO"        # 40–59 redefinido abaixo
    faixa[anos  <  40]                         = "ADULTO JOVEM"
    faixa[anos  <  18]                         = "ADOLESCENTE"
    faixa[anos  <  12]                         = "CRIANÇA"
    faixa[dias  <  365.25 * 2]                 = "LACTENTE"
    faixa[dias  <  28]                         = "NEONATO"
    faixa[unidade == 1]                        = "NEONATO"       # horas → sempre neonato
    faixa[np.isnan(raw)]                       = "NÃO INFORMADO"

    # OTIMIZAÇÃO: 8 rótulos distintos em 2,5 M de linhas. Candidato óbvio.
    df["FAIXA_ETARIA"] = pd.Categorical(faixa)
    return df

for ano in sorted(dfs.keys()):
    dfs[ano] = enriquecer_idade(dfs[ano])
gc.collect()

# Confirmação enxuta
ano_rec = max(dfs.keys())
print(f"✅ Idade enriquecida. Distribuição de FAIXA_ETARIA em {ano_rec}:")
print(dfs[ano_rec]["FAIXA_ETARIA"].value_counts(dropna=False).to_string())
mem("(após o PASSO 8b)")

## PASSO 9: Decodificação de códigos categóricos

O SINAN guarda quase todos os campos categóricos como **números** (`CS_SEXO = "M"`,
`EVOLUCAO = "3"`, `CIRCUNSTAN = "10"`). Isso é ótimo para o banco, mas **péssimo para
o Power BI**, porque ninguém entende.

Aqui criamos uma coluna `<CAMPO>_DESC` para cada código importante, traduzindo conforme
o **dicionário oficial v5 do SINAN-IEXO**. Mantemos a coluna original (útil para filtros
e merges) e adicionamos a descrição (para visualizações).

> 📖 Os mapeamentos abaixo vieram da página 4–7 do dicionário oficial v5 (link no topo
> do notebook). Se em uma versão futura algum código mudar, é só atualizar o dicionário
> `MAPS`.

In [ ]:
# Mapeamentos código → descrição, baseados no dicionário oficial SINAN-IEXO v5
MAPS = {
    "CS_SEXO": {"M": "Masculino", "F": "Feminino", "I": "Ignorado"},

    "CS_GESTANT": {
        "1": "1º Trimestre", "2": "2º Trimestre", "3": "3º Trimestre",
        "4": "Idade gestacional ignorada", "5": "Não", "6": "Não se aplica",
        "9": "Ignorado",
    },

    "CS_RACA": {
        "1": "Branca", "2": "Preta", "3": "Amarela", "4": "Parda",
        "5": "Indígena", "9": "Ignorado",
    },

    "CS_ESCOL_N": {
        "00": "Analfabeto",
        "01": "1ª-4ª série incompleta EF",
        "02": "4ª série completa EF",
        "03": "5ª-8ª série incompleta EF",
        "04": "Ensino Fundamental completo",
        "05": "Ensino Médio incompleto",
        "06": "Ensino Médio completo",
        "07": "Educação Superior incompleta",
        "08": "Educação Superior completa",
        "09": "Ignorado",
        "10": "Não se aplica",
        "99": "Ignorado",
    },

    "CS_ZONA": {"1": "Urbana", "2": "Rural", "3": "Periurbana", "9": "Ignorado"},

    # Grupo do agente tóxico (campo 49 da ficha)
    "TP_AGENTE": {
        "01": "Medicamento", "02": "Agrotóxico/uso agrícola",
        "03": "Agrotóxico/uso doméstico", "04": "Agrotóxico/uso saúde pública",
        "05": "Raticida", "06": "Produto veterinário",
        "07": "Produto de uso domiciliar", "08": "Cosmético/higiene pessoal",
        "09": "Produto químico industrial", "10": "Metal",
        "11": "Drogas de abuso", "12": "Planta tóxica",
        "13": "Alimento e bebida", "14": "Outro", "99": "Ignorado",
    },

    # Via de exposição/contaminação (campo 54)
    "VIA_1": {
        "1": "Digestiva", "2": "Cutânea", "3": "Respiratória", "4": "Ocular",
        "5": "Parenteral", "6": "Vaginal", "7": "Transplacentária",
        "8": "Outra", "9": "Ignorado",
    },
    "VIA_2": {
        "1": "Digestiva", "2": "Cutânea", "3": "Respiratória", "4": "Ocular",
        "5": "Parenteral", "6": "Vaginal", "7": "Transplacentária",
        "8": "Outra", "9": "Ignorado",
    },

    # Circunstância da exposição (campo 55) - importante para entender padrões!
    "CIRCUNSTAN": {
        "01": "Uso habitual", "02": "Acidental", "03": "Ambiental",
        "04": "Uso terapêutico", "05": "Prescrição médica inadequada",
        "06": "Erro de administração", "07": "Automedicação", "08": "Abuso",
        "09": "Ingestão de alimento/bebida", "10": "Tentativa de suicídio",
        "11": "Tentativa de aborto", "12": "Violência/homicídio",
        "13": "Outra", "99": "Ignorado",
    },

    # Tipo de exposição (campo 57)
    "TPEXP": {
        "1": "Aguda – única", "2": "Aguda – repetida", "3": "Crônica",
        "4": "Aguda sobre crônica", "9": "Ignorado",
    },

    # Tipo de atendimento (campo 59)
    "TP_ATENDE": {
        "1": "Hospitalar", "2": "Ambulatorial", "3": "Domiciliar",
        "4": "Nenhum", "9": "Ignorado",
    },

    # Houve hospitalização (campo 60)
    "HOSPITAL": {"1": "Sim", "2": "Não", "9": "Ignorado"},

    # Classificação final (campo 65)
    "CLASSI_FIN": {
        "1": "Intoxicação confirmada", "2": "Exposição",
        "3": "Reação adversa", "4": "Diagnóstico diferencial",
        "5": "Síndrome de abstinência", "9": "Ignorado",
    },

    # Evolução do caso (campo 68) - o mais importante!
    "EVOLUCAO": {
        "1": "Cura sem sequela", "2": "Cura com sequela",
        "3": "Óbito por intoxicação exógena", "4": "Óbito por outra causa",
        "5": "Perda de seguimento", "9": "Ignorado",
    },

    # Comunicação de Acidente de Trabalho - CAT (campo 70)
    "CAT": {"1": "Sim", "2": "Não", "3": "Não se aplica", "9": "Ignorado"},
}

In [ ]:
def decodificar(df: pd.DataFrame) -> pd.DataFrame:
    for col, mapa in MAPS.items():
        if col in df.columns:
            df[f"{col}_DESC"] = (
                df[col].astype("string")
                       .replace("", pd.NA)      # ← vazio vira nulo antes do map
                       .map(mapa)
                       .fillna("Não informado")
                       # OTIMIZAÇÃO: cada _DESC tem no máximo ~14 rótulos em
                       # 2,5 M de linhas. Como `string`/`object` isso custaria
                       # centenas de MB por coluna, e são 17 colunas novas.
                       .astype("category")
            )
    return df

# OTIMIZAÇÃO: laço no lugar do dict comprehension (não mantém as duas versões
# do dicionário vivas ao mesmo tempo).
for ano in sorted(dfs):
    dfs[ano] = decodificar(dfs[ano])
gc.collect()

# Verifica
ano_recente = max(dfs.keys())
print("✅ Códigos decodificados. Amostra:")
print("\nDistribuição de EVOLUCAO_DESC em", ano_recente, ":")
print(dfs[ano_recente]["EVOLUCAO_DESC"].value_counts())
print("\nDistribuição de CIRCUNSTAN_DESC em", ano_recente, ":")
print(dfs[ano_recente]["CIRCUNSTAN_DESC"].value_counts())
print()
print(f"   Peso de `dfs`: {ram_dfs(dfs):.2f} GB")
mem("(após o PASSO 9)")

## PASSO 10: Indicadores derivados

Criamos colunas calculadas que serão **muito úteis em dashboards**:

| Coluna | O que mede |
|--------|------------|
| `DIAS_SINTOMA_NOTIFIC` | Dias entre o primeiro sintoma e a notificação (atraso na notificação) |
| `DIAS_ATE_ENCERRAMENTO` | Dias entre notificação e encerramento (tempo de acompanhamento) |
| `FLAG_OBITO` | 1 se o caso evoluiu para óbito (qualquer causa), 0 caso contrário |
| `FLAG_OBITO_INTOX` | 1 se o óbito foi **especificamente** por intoxicação exógena |
| `FLAG_TENT_SUICIDIO` | 1 se a circunstância foi tentativa de suicídio |
| `ANO_NOTIFIC` / `MES_NOTIFIC` | Ano e mês da notificação (para séries temporais) |
| `TRIMESTRE_NOT` | 1, 2, 3 ou 4 (útil para análises trimestrais) |
| `ANO_MES_NOTIF` | String `YYYY-MM` — ordenável e ótimo para gráficos de série temporal |

In [ ]:
def adicionar_indicadores(df: pd.DataFrame) -> pd.DataFrame:
    # Tempos (em dias)
    # NOTA: dá para trocar por Int32 e economizar ~20 MB, mas .dt.days devolve
    # float64 quando há NaT e o CSV sairia com "5" no lugar de "5.0". Como a
    # economia é irrelevante perto do resto, fica como está para o arquivo
    # exportado continuar idêntico ao da versão anterior.
    if "DT_NOTIFIC" in df.columns and "DT_SIN_PRI" in df.columns:
        df["DIAS_SINTOMA_NOTIFIC"] = (df["DT_NOTIFIC"] - df["DT_SIN_PRI"]).dt.days
    if "DT_ENCERRA" in df.columns and "DT_NOTIFIC" in df.columns:
        df["DIAS_ATE_ENCERRAMENTO"] = (df["DT_ENCERRA"] - df["DT_NOTIFIC"]).dt.days

    # Flags (Int8 = inteiro de 8 bits — economiza memória)
    if "EVOLUCAO" in df.columns:
        df["FLAG_OBITO"]       = df["EVOLUCAO"].isin(["3", "4"]).astype("Int8")
        df["FLAG_OBITO_INTOX"] = (df["EVOLUCAO"] == "3").astype("Int8")

    if "CIRCUNSTAN" in df.columns:
        df["FLAG_TENT_SUICIDIO"] = (df["CIRCUNSTAN"] == "10").astype("Int8")

    # Decomposição temporal (essencial para Power BI)
    # (mesma decisão do bloco acima: dtype numérico preservado)
    if "DT_NOTIFIC" in df.columns:
        df["ANO_NOTIFIC"]   = df["DT_NOTIFIC"].dt.year
        df["MES_NOTIFIC"]   = df["DT_NOTIFIC"].dt.month
        df["TRIMESTRE_NOT"] = df["DT_NOTIFIC"].dt.quarter
        # OTIMIZAÇÃO: ~240 rótulos distintos ("2006-01" … "2025-12") em 2,5 M
        # de linhas. Como object seriam 2,5 M de strings separadas.
        df["ANO_MES_NOTIF"] = df["DT_NOTIFIC"].dt.strftime("%Y-%m").astype("category")

    return df

for ano in sorted(dfs):
    dfs[ano] = adicionar_indicadores(dfs[ano])
gc.collect()

# Mostra estatísticas
ano_recente = max(dfs.keys())
print(f"✅ Indicadores derivados em {ano_recente}:")
df_r = dfs[ano_recente]
print(f"   Óbitos:                  {df_r['FLAG_OBITO'].sum():,} ({df_r['FLAG_OBITO'].mean()*100:.1f}%)")
print(f"   Óbitos por intoxicação:  {df_r['FLAG_OBITO_INTOX'].sum():,}")
print(f"   Tent. suicídio:          {df_r['FLAG_TENT_SUICIDIO'].sum():,} ({df_r['FLAG_TENT_SUICIDIO'].mean()*100:.1f}%)")
print(f"   Mediana dias sintoma→notif: {df_r['DIAS_SINTOMA_NOTIFIC'].median()} dias")
mem("(após o PASSO 10)")

## PASSO 12: Tabelas de Estados e Municípios (API do IBGE)

O SINAN guarda município e estado como **códigos numéricos do IBGE** (ex.: `355030` =
São Paulo, `31` = Minas Gerais). Para os dashboards mostrarem nomes, precisamos de
tabelas de referência.

Vamos buscar **direto da API oficial do IBGE** (gratuita, sem autenticação):

- `/api/v1/localidades/estados` → 27 unidades federativas + região
- `/api/v1/localidades/municipios` → ~5.570 municípios

> ⚠️ **Bug corrigido do código anterior:** a versão original tentava usar `estados_raw`
> e `municipios_raw` sem **nunca** buscar esses dados. As células de requisição estavam
> faltando. Aqui está completo.

In [ ]:
URL_ESTADOS    = "https://servicodados.ibge.gov.br/api/v1/localidades/estados"
URL_MUNICIPIOS = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

print("⏳ Baixando estados do IBGE...")
estados_raw = requests.get(URL_ESTADOS, timeout=60).json()
print(f"   {len(estados_raw)} estados recebidos")

print("⏳ Baixando municípios do IBGE (pode demorar ~30s)...")
municipios_raw = requests.get(URL_MUNICIPIOS, timeout=120).json()
print(f"   {len(municipios_raw)} municípios recebidos")

In [ ]:
# ─────────────────────────────────────────────
# Tabela de ESTADOS
# ─────────────────────────────────────────────
df_estados = pd.DataFrame([
    {
        "ID_ESTADO":  e["id"],
        "SG_UF":      e["sigla"],
        "NM_ESTADO":  e["nome"],
        "NM_REGIAO":  e["regiao"]["nome"],
        "SG_REGIAO":  e["regiao"]["sigla"],
    }
    for e in estados_raw
])

# ─────────────────────────────────────────────
# Tabela de MUNICÍPIOS
# A API do IBGE tem DUAS estruturas (uma antiga e outra nova).
# Tentamos os dois caminhos para não perder municípios.
# ─────────────────────────────────────────────
def extrair_uf(m):
    # Caminho tradicional: microrregião → mesorregião → UF
    micro = m.get("microrregiao")
    if micro and micro.get("mesorregiao") and micro["mesorregiao"].get("UF"):
        uf = micro["mesorregiao"]["UF"]
        return uf.get("sigla"), uf.get("id")

    # Caminho novo: região-imediata → região-intermediária → UF
    imediata = m.get("regiao-imediata")
    if imediata and imediata.get("regiao-intermediaria") and imediata["regiao-intermediaria"].get("UF"):
        uf = imediata["regiao-intermediaria"]["UF"]
        return uf.get("sigla"), uf.get("id")

    return None, None

lista_municipios = []
for m in municipios_raw:
    sg_uf, id_estado = extrair_uf(m)
    lista_municipios.append({
        "ID_MUNICIPIO": m["id"],
        "NM_MUNICIPIO": m["nome"],
        "SG_UF":        sg_uf,
        "ID_ESTADO":    id_estado,
    })

df_municipios = pd.DataFrame(lista_municipios).dropna(subset=["SG_UF"])

# O SINAN usa o código de município com 6 dígitos (sem o dígito verificador do IBGE).
# Adicionamos uma coluna auxiliar para o JOIN funcionar.
df_municipios["ID_MUNICIPIO_6"] = df_municipios["ID_MUNICIPIO"].astype(str).str[:6]

print(f"✅ {len(df_estados)} estados | {len(df_municipios)} municípios carregados")
df_municipios.head()

## PASSO 13: Enriquecimento geográfico (merge com IBGE)

Agora juntamos as tabelas de estado e município com cada DataFrame de ano. Resultado:
cada linha do SINAN passa a ter o **nome completo** do município, estado e região de
residência do paciente.

> 🗺️ Usamos `ID_MN_RESI` (município **de residência** do paciente), que costuma ser
> o campo mais analisado em epidemiologia. Você pode adaptar para `ID_MUNICIP`
> (município de **notificação**) se quiser estudar onde os casos são atendidos.

In [ ]:
def enriquecer_geo(df: pd.DataFrame, df_mun: pd.DataFrame, df_est: pd.DataFrame) -> pd.DataFrame:
    if "ID_MN_RESI" not in df.columns:
        return df

    # OTIMIZAÇÃO: removido o `df = df.copy()`. Era ele o principal responsável
    # pelo OOM: copiava as ~110 colunas do ano inteiro logo antes de dois merges
    # que, por definição, já devolvem DataFrames novos. Eram três cópias
    # completas vivas ao mesmo tempo. Como o retorno é reatribuído em
    # `dfs[ano]`, mutar in-place aqui é seguro.

    # Garante string de 6 dígitos para casar com o IBGE
    df["ID_MN_RESI"] = df["ID_MN_RESI"].astype(str).str[:6]

    # OTIMIZAÇÃO/SEGURANÇA: se o lado direito do merge tivesse chave duplicada,
    # o resultado explodiria em linhas e derrubaria a RAM. O código do IBGE já é
    # único, então o drop_duplicates não muda nada — só impede o acidente.
    lookup_mun = (
        df_mun[["ID_MUNICIPIO_6", "NM_MUNICIPIO", "SG_UF"]]
        .drop_duplicates("ID_MUNICIPIO_6")
        .rename(columns={
            "ID_MUNICIPIO_6": "ID_MN_RESI",
            "NM_MUNICIPIO":   "NM_MUN_RESI",
            "SG_UF":          "SG_UF_RESI",
        })
    )
    lookup_est = (
        df_est[["SG_UF", "NM_ESTADO", "NM_REGIAO"]]
        .drop_duplicates("SG_UF")
        .rename(columns={
            "SG_UF":     "SG_UF_RESI",
            "NM_ESTADO": "NM_ESTADO_RESI",
            "NM_REGIAO": "NM_REGIAO_RESI",
        })
    )

    # JOIN 1: com municípios
    df = df.merge(lookup_mun, on="ID_MN_RESI", how="left")
    # JOIN 2: com estados (para pegar nome do estado e região)
    df = df.merge(lookup_est, on="SG_UF_RESI", how="left")

    # OTIMIZAÇÃO: as quatro colunas novas são de baixíssima cardinalidade
    # (5.571 municípios, 27 UFs, 5 regiões) em 2,5 M de linhas.
    for col in ("ID_MN_RESI", "NM_MUN_RESI", "SG_UF_RESI",
                "NM_ESTADO_RESI", "NM_REGIAO_RESI"):
        if col in df.columns:
            df[col] = df[col].astype("category")

    return df

# OTIMIZAÇÃO: laço no lugar do dict comprehension. Aqui a diferença é crítica —
# o comprehension mantinha as 20 versões antigas e as 20 novas simultaneamente.
for ano in sorted(dfs):
    dfs[ano] = enriquecer_geo(dfs[ano], df_municipios, df_estados)
    gc.collect()

# Verifica taxa de "match"
ano_recente = max(dfs.keys())
df_r = dfs[ano_recente]
pct_match = df_r["NM_MUN_RESI"].notna().mean() * 100
print(f"✅ Geografia enriquecida em {ano_recente}: {pct_match:.1f}% dos registros casaram com município")
print("\nAmostra:")
print(f"   Peso de `dfs`: {ram_dfs(dfs):.2f} GB")
mem("(após o PASSO 13)")
df_r[["ID_MN_RESI", "NM_MUN_RESI", "NM_ESTADO_RESI", "NM_REGIAO_RESI"]].head()

## PASSO 14: Validação da qualidade da base tratada

Antes de exportar, vamos **conferir o estado da base**. Para a sua IC, esses números
são ouro: mostram para os professores **o quão limpa e completa ficou a base**.

In [ ]:
# PASSO 14 — Relatório de qualidade
#
# OTIMIZAÇÃO: esta célula fazia `df_final = pd.concat(dfs.values())`, duplicando
# a base inteira na RAM (e deixando `df_final` vivo até o fim). Agora tudo é
# calculado num passe único sobre `dfs`, com acumuladores. Os números impressos
# são exatamente os mesmos; a única concatenação que sobra é a da coluna
# DIAS_SINTOMA_NOTIFIC, necessária para a mediana global (~20 MB).

anos_ord = sorted(dfs.keys())

campos_chave = [
    "DT_NOTIFIC", "DT_NASC", "CS_SEXO_DESC", "IDADE_ANOS", "FAIXA_ETARIA",
    "NM_MUN_RESI", "NM_ESTADO_RESI", "TP_AGENTE_DESC", "VIA_1_DESC",
    "CIRCUNSTAN_DESC", "EVOLUCAO_DESC",
]
flags = ["FLAG_OBITO", "FLAG_OBITO_INTOX", "FLAG_TENT_SUICIDIO"]

total       = 0
todas_cols  = []          # união das colunas, na mesma ordem que o concat daria
vistas      = set()
notna       = {c: 0 for c in campos_chave}
flag_soma   = {c: 0 for c in flags}
flag_notna  = {c: 0 for c in flags}
idade_min = idade_max = None
anos_unicos = []
dias_chunks = []
vc_agente   = None

for ano in anos_ord:
    df = dfs[ano]
    total += len(df)

    for c in df.columns:
        if c not in vistas:
            vistas.add(c)
            todas_cols.append(c)

    if "ANO_NOTIFIC" in df.columns:
        anos_unicos.append(np.asarray(df["ANO_NOTIFIC"].dropna().unique()))

    for c in campos_chave:
        if c in df.columns:
            notna[c] += int(df[c].notna().sum())

    for c in flags:
        if c in df.columns:
            flag_soma[c]  += int(df[c].sum())
            flag_notna[c] += int(df[c].notna().sum())

    if "IDADE_ANOS" in df.columns:
        mn, mx = df["IDADE_ANOS"].min(), df["IDADE_ANOS"].max()
        idade_min = mn if idade_min is None else min(idade_min, mn)
        idade_max = mx if idade_max is None else max(idade_max, mx)

    if "DIAS_SINTOMA_NOTIFIC" in df.columns:
        dias_chunks.append(df["DIAS_SINTOMA_NOTIFIC"])

    if "AGENTE_1" in df.columns:
        vc = df["AGENTE_1"].value_counts()
        vc_agente = vc if vc_agente is None else vc_agente.add(vc, fill_value=0)

anos_cobertos = np.unique(np.concatenate(anos_unicos)) if anos_unicos else np.array([])
mediana_dias  = pd.concat(dias_chunks, ignore_index=True).median() if dias_chunks else float("nan")
liberar("dias_chunks")

print("=" * 60)
print("📊 RELATÓRIO DE QUALIDADE DA BASE TRATADA")
print("=" * 60)
print(f"\n🔹 Volume")
print(f"   Total de registros: {total:,}")
print(f"   Total de colunas:   {len(todas_cols)}")
print(f"   Anos cobertos:      {sorted(anos_cobertos.astype(int))}")

print(f"\n🔹 Completude dos campos-chave (% não-nulo)")
for c in campos_chave:
    if c in vistas:
        pct = notna[c] / total * 100
        bar = "█" * int(pct/5) + "░" * (20 - int(pct/5))
        print(f"   {c:<22} {bar} {pct:5.1f}%")

def _pct(a, b):
    """Reproduz o que Series.mean() daria: nan quando não há valor não-nulo."""
    return a / b * 100 if b else float("nan")

print(f"\n🔹 Indicadores epidemiológicos")
print(f"   Óbitos totais:       {flag_soma['FLAG_OBITO']:,} ({_pct(flag_soma['FLAG_OBITO'], flag_notna['FLAG_OBITO']):.2f}%)")
print(f"   Óbitos por intox.:   {flag_soma['FLAG_OBITO_INTOX']:,} (letalidade específica)")
print(f"   Tent. de suicídio:   {flag_soma['FLAG_TENT_SUICIDIO']:,} ({_pct(flag_soma['FLAG_TENT_SUICIDIO'], flag_notna['FLAG_TENT_SUICIDIO']):.2f}%)")

print(f"\n🔹 Sanidade")
print(f"   Idade min/max:       {idade_min} / {idade_max}")
print(f"   Tempo sint→notif (mediana): {mediana_dias:.0f} dias")

print(f"\n🔹 Top 5 agentes")
top_agentes = vc_agente.astype("int64").sort_values(ascending=False).head()
top_agentes.index = top_agentes.index.astype(object)
print(top_agentes.to_string())

mem("(após o PASSO 14)")



```
# Isto está formatado como código
```

## PASSO 15: Validações Importantes antes de Exportar para o Power BI (resumo tratamentos)



In [ ]:
# PASSO 15 — Validações antes de exportar para o Power BI

import pandas as pd

erros  = 0
avisos = 0

def ok(msg):
    print(f"  ✅ {msg}")

def erro(msg, passo=None):
    global erros
    erros += 1
    ref = f" → revise o PASSO {passo}" if passo else ""
    print(f"  ❌ {msg}{ref}")

def aviso(msg):
    global avisos
    avisos += 1
    print(f"  ⚠️  {msg}")

anos_ord = sorted(dfs.keys())
sep      = "=" * 60

# ── Acumuladores (um passe único por df, sem concat) ──────────
total          = 0
n_anos         = len(anos_ord)
n_colunas      = len(dfs[anos_ord[-1]].columns)   # ano mais recente como referência

# Datas
nat_obrig      = {c: 0 for c in ["DT_NOTIFIC", "DT_SIN_PRI"]}
nat_opcio      = {c: 0 for c in ["DT_INVEST", "DT_ENCERRA", "DT_OBITO", "DT_DIGITA"]}
dtype_erros    = []
n_dt_invertida = 0

# Idade
n_idade_neg    = 0
n_idade_inv    = 0
n_idade_nat    = 0
n_faixa_ni     = 0
cols_idade_ok  = True

# Decodificações
ni_desc        = {c: 0 for c in ["CS_SEXO_DESC", "EVOLUCAO_DESC", "CIRCUNSTAN_DESC", "AGENTE_TOX_DESC"]}
cols_desc_miss = []

# Flags
flag_invalidos = {c: 0 for c in ["FLAG_OBITO", "FLAG_OBITO_INTOX", "FLAG_TENT_SUICIDIO"]}
flag_miss      = []
n_flag_incons  = 0
flag_pos       = {c: 0 for c in ["FLAG_OBITO", "FLAG_OBITO_INTOX", "FLAG_TENT_SUICIDIO"]}

# Geografia
nat_geo        = {c: 0 for c in ["NM_MUN_RESI", "NM_ESTADO_RESI", "NM_REGIAO_RESI"]}
cols_geo_miss  = []

for ano in anos_ord:
    df  = dfs[ano]
    n   = len(df)
    total += n

    # Datas obrigatórias
    for col in nat_obrig:
        if col in df.columns:
            if not pd.api.types.is_datetime64_any_dtype(df[col]):
                if col not in dtype_erros:
                    dtype_erros.append(col)
            else:
                nat_obrig[col] += int(df[col].isna().sum())

    # Datas opcionais
    for col in nat_opcio:
        if col in df.columns:
            nat_opcio[col] += int(df[col].isna().sum())

    # Consistência temporal
    if {"DT_SIN_PRI", "DT_NOTIFIC"} <= set(df.columns):
        n_dt_invertida += int((df["DT_SIN_PRI"] > df["DT_NOTIFIC"]).sum())

    # Idade
    for col in ["IDADE_ANOS", "IDADE_DESCRICAO", "FAIXA_ETARIA"]:
        if col not in df.columns:
            cols_idade_ok = False

    if "IDADE_ANOS" in df.columns:
        idade_f = df["IDADE_ANOS"].astype("float64")
        n_idade_neg += int((idade_f < 0).sum())
        n_idade_inv += int((idade_f > 130).sum())
        n_idade_nat += int(df["IDADE_ANOS"].isna().sum())

    if "FAIXA_ETARIA" in df.columns:
        n_faixa_ni += int((df["FAIXA_ETARIA"] == "NÃO INFORMADO").sum())

    # Decodificações
    for col in ni_desc:
        if col not in df.columns:
            if col not in cols_desc_miss:
                cols_desc_miss.append(col)
        else:
            ni_desc[col] += int((df[col] == "Não informado").sum())

    # Flags
    for col in flag_invalidos:
        if col not in df.columns:
            if col not in flag_miss:
                flag_miss.append(col)
        else:
            nao_nulos = df[col].dropna()
            flag_invalidos[col] += int((~nao_nulos.astype("int8").isin([0, 1])).sum())
            flag_pos[col]       += int((nao_nulos == 1).sum())

    if {"FLAG_OBITO", "FLAG_OBITO_INTOX"} <= set(df.columns):
        n_flag_incons += int(
            ((df["FLAG_OBITO"].fillna(0) == 0) & (df["FLAG_OBITO_INTOX"].fillna(0) == 1)).sum()
        )

    # Geografia
    for col in nat_geo:
        if col not in df.columns:
            if col not in cols_geo_miss:
                cols_geo_miss.append(col)
        else:
            nat_geo[col] += int(df[col].isna().sum())

# ── IMPRESSÃO DOS RESULTADOS ──────────────────────────────────

print(sep)
print("1. VOLUME")
print(sep)
print(f"  Registros : {total:>12,}")
print(f"  Anos      : {n_anos:>12,}")
print(f"  Colunas   : {n_colunas:>12,}")
ok(f"{total:,} registros carregados.") if total >= 2_000_000 else aviso(
    f"Menos de 2 M de registros ({total:,}) — confirme se todos os anos foram baixados.")

print(f"\n{sep}")
print("2. CONVERSÃO DE DATAS (PASSO 6)")
print(sep)
for col, n_nat in nat_obrig.items():
    if col in dtype_erros:
        erro(f"{col} não é datetime.", passo=6)
    elif n_nat:
        erro(f"{col}: {n_nat:,} NaT — campo obrigatório.", passo=6)
    else:
        ok(f"{col}: 100% preenchida, dtype correto.")
for col, n_nat in nat_opcio.items():
    pct = n_nat / total * 100
    msg = f"{col}: {total - n_nat:,} preenchidas ({100-pct:.1f}%), {n_nat:,} NaT ({pct:.1f}%)"
    (ok if pct < 50 else aviso)(msg)
if n_dt_invertida:
    aviso(f"{n_dt_invertida:,} registros com DT_SIN_PRI > DT_NOTIFIC.")
else:
    ok("DT_SIN_PRI ≤ DT_NOTIFIC em todos os registros.")

print(f"\n{sep}")
print("3. IDADE (PASSO 8)")
print(sep)
if not cols_idade_ok:
    erro("Uma ou mais colunas de idade ausentes (IDADE_ANOS / IDADE_DESCRICAO / FAIXA_ETARIA).", passo=8)
if n_idade_neg: erro(f"IDADE_ANOS: {n_idade_neg:,} valores negativos.", passo=8)
if n_idade_inv: aviso(f"IDADE_ANOS: {n_idade_inv:,} valores > 130 anos.")
ok(f"IDADE_ANOS: {(1 - n_idade_nat/total)*100:.1f}% preenchida{', sem negativos' if not n_idade_neg else ''}.")
pct_ni = n_faixa_ni / total * 100
(ok if pct_ni < 5 else aviso)(f"FAIXA_ETARIA: {n_faixa_ni:,} 'NÃO INFORMADO' ({pct_ni:.1f}%)")

print(f"\n{sep}")
print("4. DECODIFICAÇÕES DE CÓDIGO (PASSO 9)")
print(sep)
for col in cols_desc_miss:
    erro(f"{col} ausente.", passo=9)
for col, n_ni in ni_desc.items():
    if col in cols_desc_miss: continue
    pct = n_ni / total * 100
    (ok if pct < 10 else aviso)(f"{col}: {n_ni:,} 'Não informado' ({pct:.1f}%)")

print(f"\n{sep}")
print("5. INDICADORES DERIVADOS (PASSO 10)")
print(sep)
for col in flag_miss:
    erro(f"{col} ausente.", passo=10)
for col, n_inv in flag_invalidos.items():
    if col in flag_miss: continue
    if n_inv:
        erro(f"{col}: {n_inv:,} valores fora de {{0, 1}}.", passo=10)
    else:
        n1 = flag_pos[col]
        ok(f"{col}: {n1:,} positivos ({n1/total*100:.2f}%).")
if n_flag_incons:
    erro(f"{n_flag_incons:,} registros com FLAG_OBITO=0 mas FLAG_OBITO_INTOX=1.", passo=10)
else:
    ok("FLAG_OBITO ≥ FLAG_OBITO_INTOX em todos os registros.")

print(f"\n{sep}")
print("6. ENRIQUECIMENTO GEOGRÁFICO (PASSO 13)")
print(sep)
for col in cols_geo_miss:
    erro(f"{col} ausente.", passo=13)
for col, n_nat in nat_geo.items():
    if col in cols_geo_miss: continue
    pct = n_nat / total * 100
    (ok if pct < 5 else aviso)(f"{col}: {n_nat:,} sem match ({pct:.1f}%)")

print(f"\n{sep}")
print("RESULTADO")
print(sep)
if erros:
    print(f"  🚫 {erros} erro(s) encontrado(s). Não exporte até corrigir.")
elif avisos:
    print(f"  ⚠️  {avisos} aviso(s). Avalie antes de exportar.")
else:
    print("  🟢 Todas as validações passaram. Base pronta para exportação.")

## PASSO 15: Exportação final para Power BI

Tudo pronto. Vamos:

1. **Reordenar as colunas** de forma lógica (identificação → demografia → exposição →
   desfecho → datas)
2. **Salvar em CSV com encoding `utf-8-sig`** — isso garante que o Power BI no Windows
   abra os acentos corretamente
3. **Mostrar o caminho do arquivo** para você baixar do Colab

> 💾 **Tamanho:** o arquivo final do SINAN-IEXO inteiro (~20 anos) costuma ficar entre
> 200 e 800 MB. O Power BI lê tranquilamente.

> ⬇️ **Para baixar do Colab:** clique no ícone de pasta na barra lateral esquerda do Colab,
> ache o arquivo `.csv` e clique nos 3 pontinhos → Download. Ou rode a célula opcional
> de download direto no final.

In [ ]:
# PASSO 15 — Exportação final para Power BI
#
# OTIMIZAÇÃO: a versão anterior dependia de `df_final`, o concat de toda a base.
# Materializar 2,5 M × ~110 colunas e depois serializar para CSV era o segundo
# pico de memória do notebook. Agora o CSV é escrito ano a ano, em modo append:
# só um ano fica na RAM por vez e o arquivo gerado é byte a byte o mesmo.

ano_min, ano_max = min(dfs.keys()), max(dfs.keys())
anos_ord = sorted(dfs.keys())

# Ordem preferida (colunas que existem vão para o início; o resto fica no fim)
ordem_preferida = [
    # Identificação temporal
    "NU_ANO", "ANO_NOTIFIC", "MES_NOTIFIC", "TRIMESTRE_NOT", "ANO_MES_NOTIF",
    "DT_NOTIFIC", "DT_SIN_PRI", "DT_INVEST", "DT_ENCERRA",

    # Localização
    "SG_UF_NOT", "ID_MUNICIP",
    "ID_MN_RESI", "NM_MUN_RESI", "SG_UF_RESI", "NM_ESTADO_RESI", "NM_REGIAO_RESI",
    "CS_ZONA", "CS_ZONA_DESC",

    # Demografia
    "DT_NASC", "NU_IDADE_N", "IDADE_ANOS", "IDADE_MESES", "IDADE_DIAS",
    "IDADE_DESCRICAO", "FAIXA_ETARIA",
    "CS_SEXO", "CS_SEXO_DESC", "CS_GESTANT", "CS_GESTANT_DESC",
    "CS_RACA", "CS_RACA_DESC", "CS_ESCOL_N", "CS_ESCOL_N_DESC",

    # Exposição
    "TP_AGENTE", "TP_AGENTE_DESC", "AGENTE_1", "AGENTE_2", "AGENTE_3",
    "VIA_1", "VIA_1_DESC", "VIA_2", "VIA_2_DESC",
    "CIRCUNSTAN", "CIRCUNSTAN_DESC", "FLAG_TENT_SUICIDIO",
    "TPEXP", "TPEXP_DESC",

    # Atendimento
    "TP_ATENDE", "TP_ATENDE_DESC", "HOSPITAL", "HOSPITAL_DESC",
    "CAT", "CAT_DESC",

    # Desfecho
    "CLASSI_FIN", "CLASSI_FIN_DESC",
    "EVOLUCAO", "EVOLUCAO_DESC", "FLAG_OBITO", "FLAG_OBITO_INTOX",
    "DT_OBITO", "DIAS_SINTOMA_NOTIFIC", "DIAS_ATE_ENCERRAMENTO",
]

# União das colunas na mesma ordem que o pd.concat produziria: ordem do primeiro
# ano, depois as colunas novas conforme aparecem. O reindex por ano reproduz o
# preenchimento com nulo que o concat fazia nos anos que não têm a coluna.
todas_cols, vistas = [], set()
for ano in anos_ord:
    for c in dfs[ano].columns:
        if c not in vistas:
            vistas.add(c)
            todas_cols.append(c)

ordem_final = [c for c in ordem_preferida if c in vistas]
demais      = [c for c in todas_cols if c not in ordem_final]
colunas     = ordem_final + demais

# Salva com BOM para Excel/Power BI abrirem acentos corretamente
filename = f"sinan_iexo_{ano_min}_{ano_max}_tratado.csv"
if os.path.exists(filename):
    os.remove(filename)   # append puro duplicaria os dados numa re-execução

n_linhas = 0
for i, ano in enumerate(anos_ord):
    bloco = dfs[ano].reindex(columns=colunas)
    bloco.to_csv(
        filename,
        index=False,
        mode="w" if i == 0 else "a",
        header=(i == 0),
        # o BOM entra uma vez só, no começo do arquivo
        encoding="utf-8-sig" if i == 0 else "utf-8",
    )
    n_linhas += len(bloco)
    del bloco
    gc.collect()
    print(f"   … {ano} gravado  ({n_linhas:,} linhas acumuladas)")

print(f"\n✅ Arquivo exportado: {filename}")
print(f"   Linhas:   {n_linhas:,}")
print(f"   Colunas:  {len(colunas)}")

# Tamanho do arquivo
tamanho_mb = os.path.getsize(filename) / 1024 / 1024
print(f"   Tamanho:  {tamanho_mb:.1f} MB")
mem("(após a exportação)")

# Opcional: copiar o CSV para o Drive, para não perder no reset do runtime.
# import shutil; shutil.copy(filename, PASTA_DRIVE / filename)

### 15.1 (Opcional) Baixar o arquivo direto pelo navegador

In [ ]:
# Descomente as linhas abaixo se quiser baixar o CSV direto pelo navegador
# (em vez de pelo painel de arquivos do Colab)
#
# from google.colab import files
# files.download(filename)

In [ ]:
# Diagnóstico: nomes reais das colunas relacionadas a agente tóxico
ano_ref = max(dfs.keys())
cols_agente = [c for c in dfs[ano_ref].columns if "AGENT" in c.upper() or "TOX" in c.upper()]
print("Colunas encontradas:", cols_agente)
print()
print("Amostra de valores:")
for c in cols_agente:
    print(f"\n  {c}:")
    print(dfs[ano_ref][c].value_counts(dropna=False).head(8).to_string())